# Instalar el motor y abrir un artefacto versionado

- **Nivel:** 1, Fundamentos
- **Requisitos previos:** ninguno, este es el cuaderno de entrada
- **Tiempo de lectura:** unos 20 minutos

## Objetivos

Al terminar este cuaderno puedes:

- instalar el motor de validación `riskvalidation` y decir qué versión estás usando;
- listar el catálogo de pruebas de validación del motor y leer qué dice una entrada;
- abrir, una clave a la vez, los tres tipos de documento que Contraste versiona (el índice, el manifiesto de un caso y el artefacto de un caso);
- decir de dónde sale cada número de los cuadernos siguientes y con qué versión del motor se calculó.

## Preparación

El entorno de los cuadernos está fijado en `notebooks/requirements.txt`. Instálalo una vez, en un entorno virtual, con el comando de siempre:

```bash
pip install -r notebooks/requirements.txt
```

o instala solo el paquete que usa este cuaderno, con la misma versión exacta:

```bash
pip install riskvalidation==1.0.1
```

`scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` en Windows) construye por ti el entorno completo, `.venv-notebooks`. Un cuaderno nunca instala nada por su cuenta.

La celda siguiente imprime la versión del paquete instalado, para que la compares con la fijada.

In [1]:
# Imprimir la versión del paquete que usa este cuaderno, para compararla con notebooks/requirements.txt
import importlib.metadata as md
import json
from collections import Counter
from pathlib import Path

import riskvalidation as rv

print("riskvalidation", md.version("riskvalidation"))
print("versión del módulo:", rv.__version__)

riskvalidation 0.9.0
versión del módulo: 0.9.0


## Datos

Contraste es el banco de trabajo de un validador: ajusta modelos con datos públicos, corre sobre ellos las pruebas del motor y versiona el resultado como documentos. Este cuaderno abre esos documentos; todavía no ajusta ni prueba nada.

- **Propietario:** este repositorio, https://github.com/fsantibanezleal/CAOS_Contraste. Los documentos los escribe el pipeline sin conexión (`data-pipeline/`) con el motor, y aquí se leen tal como están versionados.
- **Los tres tipos de documento** (contrato 2): el **índice** `data/derived/manifests/index.json` lista los casos; un **manifiesto** `data/derived/manifests/<caso>.json` describe un caso y lista sus artefactos; un **artefacto** `data/derived/<caso>/<variante>.json` guarda los modelos, las filas de pruebas, el impacto y los hallazgos de una variante del caso.
- **El caso que abrimos, C01:** PD de tarjetas de crédito minoristas. Sus datos reales son el archivo de tarjetas de crédito de Taiwán del repositorio UCI (Yeh, 2009, 30.000 tarjetas), clase de licencia `mirror-allowed`, CC BY 4.0, https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients. El segundo caso que miramos, C22, es sintético: mide las pruebas sobre generadores con verdad conocida.
- **Archivos que se leen:** `data/derived/manifests/index.json`, `data/derived/manifests/C01.json`, `data/derived/C01/holdout.json` (y, para la procedencia del paso 5, los otros nueve documentos que lista el manifiesto) y `data/derived/C22/null.json`. Toda ruta es relativa a la raíz del repositorio y no hace falta ningún dato crudo.

La celda siguiente encuentra la raíz del repositorio (el cuaderno está tres carpetas por debajo) y define un lector que se detiene con la ruta del archivo cuando falta uno.

In [2]:
# El cuaderno está tres carpetas bajo la raíz del repositorio; un archivo ausente detiene el cuaderno y nombra su ruta
ROOT = Path("../../..").resolve()
DERIVED = ROOT / "data" / "derived"


def read_json(relative):
    return json.loads((DERIVED / relative).read_text(encoding="utf-8"))


print("carpeta derivada encontrada:", DERIVED.is_dir())

carpeta derivada encontrada: True


## Método

### Paso 1: el catálogo de pruebas

El motor registra cada prueba de validación con su identificador, su familia, su hipótesis nula (H0), su fuente primaria y el tipo de decisión que respalda. `catalogue()` devuelve el registro, de modo que un validador puede preguntar qué existe antes de correr nada. La celda siguiente cuenta las pruebas por familia y muestra una entrada completa. Los identificadores, las claves de los documentos y los textos propios del motor (una hipótesis nula, una referencia, una nota) se imprimen tal como el motor los escribe, en inglés, en ambos árboles.

In [3]:
# Listar el registro e imprimir una entrada completa
from riskvalidation import catalogue

tests = catalogue()
print("pruebas registradas:", len(tests))
for family, count in sorted(Counter(t.family for t in tests).items()):
    print(f"{family:16s} {count}")

entry = next(t for t in tests if t.test_id == "pd.jeffreys")
print()
print("test_id:", entry.test_id, "| familia:", entry.family, "| decisión:", entry.decision)
print("nombre:", entry.name)
print("H0:", entry.h0)
print("referencia:", entry.reference)

pruebas registradas: 134
aggregation      8
ccr              8
discrimination   4
fairness         12
governance       5
ifrs9            9
irrbb            7
lgd_ead          6
liquidity        11
market           12
ml               13
oprisk           5
pd-calibration   11
portfolio        5
rating-system    3
stability        5
stress           6
transitions      4

test_id: pd.jeffreys | familia: pd-calibration | decisión: p_value
nombre: Jeffreys test
H0: the PD applied at the start of the period is greater than or equal to the true PD
referencia: ECB (2019), Instructions for reporting the validation results of internal models: IRB Pillar I models for credit risk, section 2.5.3.1; Brown, Cai and DasGupta (2001), doi:10.1214/ss/1009213286


### Paso 2: el índice

El índice es la tabla de contenidos de los documentos versionados: cuántos casos hay, cuál abre el sitio por defecto y, para cada caso, su título, su categoría y la ruta de su manifiesto. La celda siguiente lo lee y agrupa los casos por categoría.

In [4]:
# Leer el índice y agrupar los casos por categoría y tipo
index = read_json("manifests/index.json")
print("esquema:", index["schema"])
print("casos:", index["n_cases"], "| caso por defecto:", index["default_case"])
print("tipos:", dict(Counter(c["kind"] for c in index["cases"])))
print("categorías:", dict(Counter(c["category_id"] for c in index["cases"])))
print("engine_version del índice igual al archivo VERSION (la versión del producto, no la de riskvalidation):",
      index["engine_version"] == (ROOT / "VERSION").read_text(encoding="utf-8").strip())
for case in index["cases"][:4]:
    print(case["case_id"], "|", case["category"]["es"], "|", case["title"]["es"], "|", case["title"]["en"])

esquema: contraste.index/v1
casos: 22 | caso por defecto: C01
tipos: {'real': 15, 'synthetic': 7}
categorías: {'credit-scoring': 3, 'ratings-calibration': 2, 'provisions': 2, 'lgd-ead': 2, 'portfolio-capital': 1, 'market-ccr': 3, 'balance-sheet': 3, 'operational': 1, 'capital-stress': 2, 'ml-ai-risk': 2, 'validator': 1}
engine_version del índice igual al archivo VERSION (la versión del producto, no la de riskvalidation): True
C01 | Scoring de crédito | PD de tarjetas: campeón vs retador | Retail cards PD: champion vs challenger
C02 | Scoring de crédito | PD corporativa a horizontes de 1 a 5 años | Corporate PD across 1 to 5 year horizons
C03 | Scoring de crédito | PD de pymes por cosecha a través de 2008 | SME PD by vintage through 2008
C04 | Calificaciones y calibración | Transiciones de calificación y PD TTC por grado | Rating transitions and TTC PD by grade


### Paso 3: el manifiesto de un caso

Un manifiesto dice qué pregunta un caso, qué fuentes lee (con su clase de licencia y su atribución), la semilla, las versiones de motor que lo produjeron y los artefactos que lista con sus tamaños. También lleva el informe del contrato 1: cuántos registros de entrada se aceptaron, se marcaron o se rechazaron. La celda siguiente lee el manifiesto de C01 e imprime los campos que un lector necesita primero.

In [5]:
# Leer el manifiesto de C01 e imprimir su pregunta, fuentes, semilla y artefactos
manifest = read_json("manifests/C01.json")
print("claves:", list(manifest))
print("pregunta:", manifest["question"]["es"])
print("semilla:", manifest["seed"], "| variante por defecto:", manifest["default_variant"])
for source in manifest["sources"]:
    detail = manifest["source_details"][source]
    print(source, "| clase:", detail["class"], "| atribución:", detail["attribution"])
print("contrato 1 sobre el archivo de Taiwán:", manifest["contract"]["uci-taiwan"]["counts"])
for item in manifest["artifacts"]:
    print(f'{item["role"]:8s} {item["variant_id"]:18s} {item["path"]:28s} {item["bytes"]:>8d} bytes')

claves: ['schema', 'case_id', 'title', 'category', 'question', 'sources', 'source_details', 'seed', 'engine', 'default_variant', 'artifacts', 'contract', 'expect']
pregunta: ¿Iguala una scorecard WoE monótona a los retadores que según la literatura pueden superarla, cuando se exige calibración, y qué dicen las pruebas de validación cuando la población cambia?
semilla: 42 | variante por defecto: holdout
uci-taiwan | clase: mirror-allowed | atribución: Yeh, I-C. (2009). Default of Credit Card Clients. UCI Machine Learning Repository. https://doi.org/10.24432/C55S3H
uci-german | clase: mirror-allowed | atribución: Hofmann, H. (1994). Statlog (German Credit Data). UCI Machine Learning Repository. https://doi.org/10.24432/C5NC77
contrato 1 sobre el archivo de Taiwán: {'input': 30000, 'accepted': 30000, 'rejected': 0, 'flagged': 26284, 'excluded': 0}
models   models-taiwan      C01/models-taiwan.json         122630 bytes
models   models-german      C01/models-german.json          48495 bytes

### Paso 4: un artefacto

Un artefacto es donde viven los números. Su envoltura tiene las mismas claves en todos los casos: los registros de `model` (los peldaños de la escalera), los `outputs` que dibuja la web, los `tests` (una fila por llamada a una prueba del motor, cada una un `TestResult` escrito como diccionario), el `impact`, los `findings` que escribiría un validador, la `provenance` y la `lane`. La celda siguiente abre el artefacto holdout de C01 y cuenta lo que contiene.

In [6]:
# Abrir el artefacto holdout de C01 y contar sus partes
holdout = read_json("C01/holdout.json")
print("claves:", list(holdout))
print("esquema:", holdout["schema"], "| caso:", holdout["case_id"], "| variante:", holdout["variant_id"])
print("peldaños:", [rung["rung"] for rung in holdout["model"]])
print("tarjetas retenidas:", holdout["outputs"]["n"], "| incumplimientos:", holdout["outputs"]["defaults"])
print("filas de prueba:", len(holdout["tests"]), "| hallazgos:", len(holdout["findings"]))
print("luces:", dict(Counter(row["light"] for row in holdout["tests"])))
print("claves de una fila de prueba:", list(holdout["tests"][0]))
print("claves de impacto:", list(holdout["impact"]))

claves: ['schema', 'case_id', 'variant_id', 'model', 'outputs', 'tests', 'impact', 'findings', 'provenance', 'lane']
esquema: contraste.case/v1 | caso: C01 | variante: holdout
peldaños: ['P0', 'P0', 'P1', 'P2', 'P2', 'P3', 'P4', 'P4']
tarjetas retenidas: 9000 | incumplimientos: 1991
filas de prueba: 187 | hallazgos: 4
luces: {'not_evaluated': 31, 'green': 136, 'red': 12, 'amber': 8}
claves de una fila de prueba: ['test_id', 'model_id', 'segment', 'statistic', 'p_value', 'metric', 'h0', 'alternative', 'policy_version', 'alpha_amber', 'alpha_red', 'light', 'n', 'n_events', 'inputs_hash', 'reference', 'notes', 'extras']
claves de impacto: ['approval_rate', 'bad_rate_champion', 'bad_rate_challenger', 'el_champion', 'el_challenger', 'loss_champion', 'loss_challenger', 'swap_in', 'swap_out', 'swap_in_bad_rate', 'swap_out_bad_rate', 'capital_champion', 'capital_challenger']


### Paso 5: la procedencia y el motor que lo calculó

Todo artefacto dice cómo se hizo: el estado de verdad de sus datos (`real-outcomes` o `synthetic-known-truth`), la semilla y la versión de `riskvalidation` que calculó sus pruebas. Los documentos versionados se calcularon en momentos distintos con versiones distintas del motor, y el cuaderno corre la versión que instalaste. Por eso, cada vez que un cuaderno posterior recalcula un número versionado, imprime la versión con que se calculó junto a la instalada; un cuaderno nunca reescribe un documento versionado. La celda siguiente las compara para todos los documentos de C01.

In [7]:
# Comparar la versión del motor que calculó cada documento de C01 con la instalada
provenance = holdout["provenance"]
print("estado de verdad:", provenance["truth_status"], "| semilla:", provenance["seed"])
print("riskvalidation instalado:", rv.__version__)

baked = Counter(read_json(item["path"])["provenance"]["riskvalidation_version"] for item in manifest["artifacts"])
print("versiones de riskvalidation que calcularon los documentos de C01:", dict(baked))
print("documentos calculados con la versión instalada:", baked.get(rv.__version__, 0), "de", len(manifest["artifacts"]))

estado de verdad: real-outcomes | semilla: 42
riskvalidation instalado: 0.9.0
versiones de riskvalidation que calcularon los documentos de C01: {'0.4.3': 10}
documentos calculados con la versión instalada: 0 de 10


### Paso 6: un segundo tipo de artefacto

C22 mide las pruebas de validación mismas, sobre generadores cuya verdad es conocida. Su artefacto tiene la misma envoltura, pero el contenido de `outputs` es otro y sus datos son sintéticos por construcción. La celda siguiente abre su variante de hipótesis nula, donde cada prueba enfrenta un modelo correcto, y compara las dos envolturas.

In [8]:
# Abrir un artefacto de C22 y comparar su envoltura con la de C01
null = read_json("C22/null.json")
print("misma envoltura que C01:", list(null) == list(holdout))
print("estado de verdad:", null["provenance"]["truth_status"], "| generadores:", null["provenance"]["generators"])
print("claves de outputs de C22:", list(null["outputs"]))
print("claves de outputs de C01:", list(holdout["outputs"]))
print("filas de prueba:", len(null["tests"]), "| primera:", null["tests"][0]["test_id"], "sobre", null["tests"][0]["model_id"])

misma envoltura que C01: True
estado de verdad: synthetic-known-truth | generadores: ['binormal_scores', 'default_counts', 'grade_frequencies', 'logistic_portfolio', 'migrations', 'rating_paths', 'two_sample']
claves de outputs de C22: ['kind', 'family', 'ladder', 'levels', 'panels', 'simulations', 'generators', 'golden', 'exact_curves', 'estimators', 'specimen', 'size_bounds']
claves de outputs de C01: ['rungs', 'irb', 'n', 'defaults', 'groups', 'sample']
filas de prueba: 5 | primera: pd.hosmer_lemeshow sobre logistic-5000


## Resultados

Para cerrar el ciclo, la celda siguiente lista las filas de pruebas a nivel de cartera de la scorecard de C01 tal como las guarda el artefacto: qué prueba, qué valor y qué luz. Estos son los números que los cuadernos siguientes recalculan con el motor instalado.

In [9]:
# Listar las filas versionadas a nivel de cartera de la scorecard P1: valor (valor p si la prueba lo tiene, si no la métrica) y luz
rows = [row for row in holdout["tests"] if row["model_id"] == "P1-scorecard" and row["segment"] == "portfolio"]
for row in rows:
    value = row["p_value"] if row["p_value"] is not None else row["metric"]
    shown = "ninguno" if value is None else format(value, ".6g")
    kind = "valor p" if row["p_value"] is not None else "métrica"
    print(f'{row["test_id"]:22s} {kind:8s} {shown:>12s}  {row["light"]}')

disc.auc               métrica      0.758722  not_evaluated
disc.auc_vs_initial    valor p       0.22574  green
disc.ks                valor p  8.14171e-220  not_evaluated
pd.jeffreys            valor p      0.309521  green
pd.binomial            valor p      0.314017  green
pd.binomial_vasicek    valor p      0.454241  green
pd.chi2_grades         valor p     0.0264445  amber
pd.hosmer_lemeshow     valor p     0.0150003  amber
pd.spiegelhalter       valor p     0.0218585  amber
pd.brier               métrica      0.139272  not_evaluated
pd.ece                 métrica     0.0155007  not_evaluated
stability.psi          valor p      0.781227  green
rating.hhi             valor p      0.497263  green


Cómo leer la tabla. La mayoría de las filas lleva un valor p y una luz (las pruebas de nivel y de ajuste). Unas pocas muestran la luz `not_evaluated`: el AUC, el puntaje de Brier y el ECE son métricas descriptivas, el estadístico de Kolmogorov-Smirnov mide separación, y la política por defecto no tiene umbral para ninguno de ellos. La luz es, entonces, la lectura que hace la política de un valor p, nunca una propiedad del modelo por sí sola; el cuaderno 03 de este tema muestra cómo la política convierte un valor p en un color. Cada fila nombra además su hipótesis nula y su fuente, de modo que un número siempre se puede rastrear hasta la definición que lo produjo.

Lo que esta tabla no muestra: es un solo modelo (la scorecard) sobre una sola población (las 9.000 tarjetas reservadas), y la calculó la versión del motor que indica la procedencia, no necesariamente la que instalaste.

## Resumen

- `riskvalidation` 0.8.1 se instala con las versiones fijadas en `notebooks/requirements.txt`, y un cuaderno imprime las versiones con que corre.
- El catálogo del motor es la lista de lo que un validador puede correr; cada entrada lleva su hipótesis nula y su fuente.
- Contraste versiona tres tipos de documento: el índice, un manifiesto por caso y un artefacto por variante, con una misma envoltura para todos los casos.
- Todo artefacto declara su procedencia, incluida la versión del motor que lo calculó, y un cuaderno compara esa versión con la instalada antes de afirmar que dos números coinciden.

**Qué probar a continuación.** Abre otro artefacto de C01 (por ejemplo `data/derived/C01/drift-severe.json`) y compara sus luces con las del holdout. Luego sigue con el cuaderno 02, en el que llamas tú mismo a una prueba sobre una porción real de 50 tarjetas.

## Referencias

- ECB (2019), Instructions for reporting the validation results of internal models: IRB Pillar I models for credit risk. https://www.bankingsupervision.europa.eu/activities/internal_models/shared/pdf/instructions_validation_reporting_credit_risk.en.pdf
- Yeh, I-C. (2009), Default of Credit Card Clients, UCI Machine Learning Repository (CC BY 4.0). https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
- El catálogo del motor y los contratos de documentos: `docs/contracts/engine/` y `docs/contracts/` de este repositorio.